<a href="https://colab.research.google.com/github/songnee/test/blob/main/4_MultipleLR.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## 다중 선형 회귀 (Multiple Linear Regression)

<img src="https://velog.velcdn.com/images/newnew_daddy/post/5b5444d9-c6eb-4744-b7b9-34026868778d/image.png" width=80%>

#### 1. 개념
- **정의**: 다중 선형 회귀는 여러 개의 독립 변수와 하나의 종속 변수 간의 관계를 모델링하는 회귀 분석 방법
- **목적**: 여러 독립 변수를 사용하여 종속 변수의 값을 예측하고, 각 독립 변수가 종속 변수에 미치는 영향을 이해
- **사용 상황**
  - 여러 개의 독립 변수가 종속 변수에 영향을 미치는 경우 사용
  - 다중 회귀 모델은 다수의 입력 변수가 종속 변수에 직선 관계로 영향을 미친다고 가정
- **예시**
  - 집 가격 예측에서, 크기, 위치, 방의 개수, 건축 연도 등의 여러 요인을 고려할 때.
  - 학생의 학업 성취도 예측에서, 수업 참여도, 과제 점수, 시험 점수, 출석률 등 여러 요인을 분석할 때.

#### 2. 수학적 표현식
$$y = \beta_0 + \beta_1x_1 + \beta_2x_2 + \cdots + \beta_nx_n + \epsilon $$

#### 3. 특징
- **선형성**: 독립 변수와 종속 변수 간의 관계가 선형으로 가정됨.
- **단순성**: 모델이 단순하고 해석이 용이함.
- **다중 공선성**: 독립 변수들 간의 상관관계가 높을 경우 다중 공선성이 발생할 수 있으며, 이는 회귀 계수의 불안정성을 초래함.

#### 1. 데이터 load

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

df = pd.read_csv('./dataset/50_Startups.csv')

df.head()

,R&D Spend,Administration,Marketing Spend,State,Profit
0,165349.20,136897.80,471784.10,New York,192261.83
1,162597.70,151377.59,443898.53,California,191792.06
2,153441.51,101145.55,407934.54,Florida,191050.39
3,144372.41,118671.85,383199.62,New York,182901.99
4,142107.34,91391.77,366168.42,Florida,166187.94


#### 2. 학습데이터(X) - 타겟데이터(y) 분리

In [ ]:
X = df[df.columns[:-2]] ## 앞 쪽 3개의 수치형 데이터만 사용
y = df[df.columns[-1]]

X.shape, y.shape

((50, 3), (50,))

In [ ]:
X.skew()

R&D Spend          0.164002
Administration    -0.489025
Marketing Spend   -0.046472
dtype: float64

In [ ]:
from sklearn.preprocessing import PowerTransformer

pt = PowerTransformer(method='box-cox') # box-cox
yj_ = pt.fit_transform(df['Administration'].to_frame())

X['Administration'] = yj_

X.skew()

C:\Users\YEIN\AppData\Local\Temp\ipykernel_442844\1422223135.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  X['Administration'] = yj_


R&D Spend          0.164002
Administration    -0.060649
Marketing Spend   -0.046472
dtype: float64

#### 3. Train Test Split

In [ ]:
from sklearn.preprocessing import StandardScaler

ss = StandardScaler()
X_scaled = ss.fit_transform(X)

In [ ]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size = 0.25, random_state = 20)

X_train.shape, X_test.shape, y_train.shape, y_test.shape

((37, 3), (13, 3), (37,), (13,))

#### 4. 모델 학습

In [ ]:
from sklearn.linear_model import LinearRegression, SGDRegressor

lr = SGDRegressor(
    max_iter=100,
    tol=0.001,
    penalty=None,
    eta0=0.1,
    random_state=42
)

# lr = LinearRegression()

lr.fit(X_train, y_train)

LinearRegression()

In [ ]:
print("Train : ", lr.score(X_train, y_train))
print("Test : ", lr.score(X_test, y_test))

Train :  0.9495958123909851
Test :  0.9479697563573677


#### 5. 결과 예측

In [ ]:
y_pred = lr.predict(X_test)

y_pred

array([[103901.8969696 ],
       [132763.05993126],
       [133567.90370044],
       [ 72911.78976736],
       [179627.92567224],
       [115166.64864795],
       [ 67113.5769057 ],
       [ 98154.80686776],
       [114756.11555221],
       [169064.01408795]])

#### 5. 모델 평가

In [ ]:
from utils import evaluate_reg_model

evaluate_reg_model(y_test, y_pred)

MAE: 7320.441614848125
MAPE: 6.288208342834581
MSE: 77506468.16885418
RMSE: 8803.775790469346
R2 Score: 0.939395591782057


#### 6. 범주형 지표 인코딩 후 모델 학습

In [ ]:
# Importing the libraries
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

# Importing the dataset
df = pd.read_csv('./dataset/50_Startups.csv')

In [ ]:
df_encoded = pd.get_dummies(df, columns=['State'], prefix='State')

df_encoded.head()

,R&D Spend,Administration,Marketing Spend,Profit,State_California,State_Florida,State_New York
0,165349.20,136897.80,471784.10,192261.83,False,False,True
1,162597.70,151377.59,443898.53,191792.06,True,False,False
2,153441.51,101145.55,407934.54,191050.39,False,True,False
3,144372.41,118671.85,383199.62,182901.99,False,False,True
4,142107.34,91391.77,366168.42,166187.94,False,True,False


In [ ]:
df_encoded.replace({False: 0, True: 1}, inplace=True)

df_encoded.head()

,R&D Spend,Administration,Marketing Spend,Profit,State_California,State_Florida,State_New York
0,165349.20,136897.80,471784.10,192261.83,0,0,1
1,162597.70,151377.59,443898.53,191792.06,1,0,0
2,153441.51,101145.55,407934.54,191050.39,0,1,0
3,144372.41,118671.85,383199.62,182901.99,0,0,1
4,142107.34,91391.77,366168.42,166187.94,0,1,0


In [ ]:
X = df_encoded.drop(['Profit'], axis=1)
y = df_encoded['Profit'].to_frame()

X.shape, y.shape

((50, 6), (50, 1))

In [ ]:
from sklearn.preprocessing import StandardScaler

ss = StandardScaler()
X_scaled = ss.fit_transform(X)

In [ ]:
# Splitting the dataset into the Training set and Test set
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.25, random_state = 20)

X_train.shape, X_test.shape, y_train.shape, y_test.shape

((37, 6), (13, 6), (37, 1), (13, 1))

In [ ]:
# Training the Multiple Linear Regression model on the Training set
from sklearn.linear_model import LinearRegression, SGDRegressor

regressor = LinearRegression()

# regressor = SGDRegressor(
#     max_iter=100
# )
regressor.fit(X_train, y_train)

LinearRegression()

In [ ]:
print("Train : ", regressor.score(X_train, y_train))
print("Test : ", regressor.score(X_test, y_test))

Train :  0.9504331001678554
Test :  0.9415825379172027


In [ ]:
y_pred = regressor.predict(X_test)

y_pred

array([[131516.6103012 ],
       [152427.42522676],
       [176383.50732747],
       [152027.23805147],
       [ 44517.47040652],
       [191168.96550645],
       [101367.23817014],
       [112839.29268002],
       [ 43670.60682686],
       [111779.28099849],
       [ 73048.33915921],
       [129715.72029562],
       [131461.07884816]])

In [ ]:
from utils import evaluate_reg_model

evaluate_reg_model(y_test, y_pred)

MAE :  7116.700816031244
MAPE :  0.06996056444695499
MSE :  90631812.130182
RMSE :  9520.074166212256
R2 :  0.9415825379172027


In [ ]:
# 계수와 상수 출력
np.set_printoptions(suppress=True)

var = regressor.feature_names_in_
coef = regressor.coef_
intercept = regressor.intercept_


print(f"종속변수: {var}")
print(f"계수: {coef}")
print(f"상수: {intercept}")

종속변수: ['R&D Spend' 'Administration' 'Marketing Spend' 'Profit'
 'State_California' 'State_Florida' 'State_New York']
계수: [[ 0.  0. -0.  1. -0.  0. -0.]]
상수: [0.]


[당뇨 수치 예측하기](https://www.codeit.kr/topics/elementary-supervised-algorithms/lessons/3085)

In [ ]:
# 필요한 라이브러리 import
from sklearn import datasets
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error

import pandas as pd

# 당뇨병 데이터 갖고 오기
diabetes_dataset = datasets.load_diabetes()

# 입력 변수를 사용하기 편하게 pandas dataframe으로 변환
X = pd.DataFrame(diabetes_dataset.data, columns=diabetes_dataset.feature_names)

# 목표 변수를 사용하기 편하게 pandas dataframe으로 변환
y = pd.DataFrame(diabetes_dataset.target, columns=['diabetes'])

# 여기에 코드를 작성하세요
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state=5)

model = LinearRegression()
model.fit(X_train, y_train)

model.coef_, model.intercept_

(array([[   2.72308829, -255.94291747,  522.84096403,  353.09406901,
         -827.60149738,  543.34104068,  115.94257227,  214.6877495 ,
          694.94897032,   32.73339672]]),
 array([152.22190213]))

In [ ]:
print("Train : ", model.score(X_train, y_train))
print("Test : ", model.score(X_test, y_test))

Train :  0.5115517387428321
Test :  0.5271558947230808


In [ ]:
y_test_pred = model.predict(X_test)

mean_squared_error(y_test, y_test_pred) ** 0.5

54.603912902946895